# A visual walkthrough of joined letters in IAM handwriting

This notebook follows a small number of handwritten IAM lines from beginning to end. It uses the real DTLR and POC files, but explains each stage in everyday language. Nothing here changes DTLR, TVA, or the frozen connectivity method.

Before running it, set the paths in the next cell. The notebook expects a completed small DTLR detection export; it can optionally run that export on a Linux/WSL machine with the RTX 4060.

In [ ]:
from pathlib import Path
import json
import os
import sys

# Change these three paths for the demonstration machine.
# POC_ROOT is the checkout containing poc/ (the dtlr-iam-bigram-poc branch).
POC_ROOT = Path(os.environ.get('DTLR_POC_ROOT', Path.cwd())).resolve()
DATA_ROOT = Path(os.environ.get('DTLR_DATA_ROOT', '/absolute/path/to/dtlr-data')).expanduser()
OUTPUT_ROOT = Path(os.environ.get('DTLR_OUTPUT_ROOT', '/absolute/path/to/dtlr-output')).expanduser()
RUN_NAME = 'iam-test-small'
DETECTIONS = OUTPUT_ROOT / RUN_NAME / 'detections.jsonl'

assert (POC_ROOT / 'poc' / 'dtlr_poc').is_dir(), 'Set DTLR_POC_ROOT to the POC checkout.'
sys.path.insert(0, str(POC_ROOT / 'poc'))
print('POC checkout:', POC_ROOT)
print('Data root:', DATA_ROOT)
print('Detection file:', DETECTIONS)

## 1. Start with a small, fixed set of lines

We use a small set so every result can be inspected. IAM supplies the correct text. DTLR is only used to estimate where each character is on the image. For a formal validation run, IDs are frozen before images are inspected; the command below creates that frozen list.

In [ ]:
# This command is shown rather than run automatically. It freezes 32 validation IDs.
print(f'''cd {POC_ROOT}
python poc/scripts/freeze_iam_selection.py \
  --data-root {DATA_ROOT} --split valid --count 32 \
  --seed iam-dominant-core-v3-20260821 \
  --output {OUTPUT_ROOT}/iam-valid-32/selection.json''')

# For this live walkthrough we normally reuse an already completed small export.
# Its records include the line ID, IAM transcription, DTLR boxes, scores, and run provenance.

## 2. Read the DTLR results

Each result contains the line image, the correct transcription, and DTLR's estimated character boxes. If the file is missing, run the printed command on the supported Linux/WSL RTX machine first.

In [ ]:
if not DETECTIONS.exists():
    raise FileNotFoundError(f'''No detections at {DETECTIONS}. Run:
python {POC_ROOT}/poc/scripts/export_iam_detections.py \
  --data-root {DATA_ROOT} \
  --checkpoint $DTLR_WEIGHTS_ROOT/finetuned/IAM/checkpoint.pth \
  --checkpoint-kind iam-finetuned --split test --start 0 --limit 8 \
  --threshold 0.3 --nms 0.5 --output {DETECTIONS}
''')

records = [json.loads(line) for line in DETECTIONS.read_text(encoding='utf-8').splitlines() if line]
print(f'Loaded {len(records)} handwritten lines.')
for record in records:
    print(f"{record['line_id']}: {record['transcription']}")

## 3. Look at one line and DTLR's estimated letter locations

The green rectangles are DTLR's best guesses for character locations. They are useful guides, but not perfect borders around the letters. That is why the later ink check is needed.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image

record = records[0]  # Change this number to present another line.
image = Image.open(DATA_ROOT / record['image_relpath']).convert('RGB')
fig, ax = plt.subplots(figsize=(16, 4))
ax.imshow(image)
for number, detection in enumerate(sorted(record['detections'], key=lambda item: item['box_xyxy'][0])):
    x0, y0, x1, y1 = detection['box_xyxy']
    ax.add_patch(Rectangle((x0, y0), x1-x0, y1-y0, fill=False, edgecolor='lime', linewidth=1))
    ax.text(x0, max(0, y0-3), str(number), color='lime', fontsize=8, backgroundcolor='black')
ax.set_title(f"IAM says: {record['transcription']}")
ax.axis('off');
plt.show()

## 4. Match the correct letters to the estimated boxes

The correct text may not line up perfectly with DTLR's predictions. A fixed matching procedure lines them up as well as possible. A mismatch remains visible instead of being hidden. Only pairs where both letters have a trustworthy matched box can be assessed.

In [ ]:
from dtlr_poc.alignment import gt_detection_map

detections = sorted(record['detections'], key=lambda item: item['box_xyxy'][0])
mapping = gt_detection_map(record['transcription'], [item['predicted_char'] for item in detections])
for index, character in enumerate(record['transcription']):
    item = mapping[index]
    box = 'no box' if item.detection_index is None else f'box {item.detection_index}'
    print(f'{index:>2}: {character!r:>4} → {box:>8} ({item.operation})')

## 5. Turn the writing into separate ink shapes

We change the gray image into a simple black-and-white ink picture. Then we colour every separate touching ink shape differently. Diagonal touching counts as touching. This gives us a direct, physical view of the handwriting.

In [ ]:
import numpy as np
from dtlr_poc.ccl import label_ink, otsu_threshold, pair_component_evidence

gray = np.asarray(image.convert('L'))
threshold = otsu_threshold(gray)
labels, component_count = label_ink(gray, threshold)
print(f'Automatic ink threshold: {threshold}; separate ink shapes: {component_count}')
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
axes[0].imshow(gray, cmap='gray'); axes[0].set_title('Original handwriting'); axes[0].axis('off')
axes[1].imshow(np.ma.masked_where(labels == 0, labels), cmap='tab20')
axes[1].set_title('Each touching ink shape has its own colour'); axes[1].axis('off')
plt.show()

## 6. Ask one simple question about two neighbouring letters

Choose a pair below. The final method gives each letter its own non-overlapping inner area. It calls the pair joined only when the same *main* ink shape is largest in both areas. If the picture is unclear, the answer is `unknown`, not `separate`.

In [ ]:
left_index = 0  # Change to inspect another neighbouring pair.
right_index = left_index + 1
left, right = mapping[left_index], mapping[right_index]
if left.detection_index is None or right.detection_index is None:
    print('Unknown: one of these letters has no matched DTLR box.')
else:
    result = pair_component_evidence(labels, detections[left.detection_index]['box_xyxy'], detections[right.detection_index]['box_xyxy'])
    pair = record['transcription'][left_index:right_index+1]
    answer = 'unknown' if not result['dominant_core_usable'] else ('joined' if result['connected_dominant_core_v3'] else 'separate')
    print(f"Pair {pair!r}: {answer}")
    print('Old full-box answer:', result['connected_box_intersection_v1'])
    print('Final dominant-core-v3 answer:', result['connected_dominant_core_v3'])
    print('If unknown, the mechanical reason is:', result['unusable_reason_codes'])

## 7. See why the final rule is safer

The old rule only asked whether any coloured ink shape appeared in both full boxes. That could be fooled by overlapping boxes. The final rule uses the two smaller inner areas shown below and asks whether the same colour is the biggest contributor on both sides.

In [ ]:
if left.detection_index is not None and right.detection_index is not None:
    fig, ax = plt.subplots(figsize=(16, 4))
    ax.imshow(np.ma.masked_where(labels == 0, labels), cmap='tab20')
    for box, colour, name in [(detections[left.detection_index]['box_xyxy'], 'deepskyblue', 'left box'),
                              (detections[right.detection_index]['box_xyxy'], 'orange', 'right box'),
                              (result['left_core_box'], 'blue', 'left inner area'),
                              (result['right_core_box'], 'red', 'right inner area')]:
        x0, y0, x1, y1 = box
        ax.add_patch(Rectangle((x0, y0), x1-x0, y1-y0, fill=False, edgecolor=colour, linewidth=2, label=name))
    ax.legend(loc='upper right'); ax.axis('off'); plt.show()
    print('Largest shape in left inner area:', result['left_dominant_component'])
    print('Largest shape in right inner area:', result['right_dominant_component'])

## 8. Repeat this for the small batch and summarize it

The pipeline repeats the same careful check for every neighbouring pair, writes one evidence row per pair, and then creates a separate score for each pair of letters. The following cell runs that evidence step if it has not already been done.

In [ ]:
EVIDENCE_DIR = OUTPUT_ROOT / RUN_NAME / 'bigrams-dominant-core-v3'
SCORES = EVIDENCE_DIR / 'bigram_scores.json'
if not SCORES.exists():
    import subprocess
    subprocess.run([sys.executable, str(POC_ROOT / 'poc/scripts/build_bigram_evidence.py'),
                    '--detections', str(DETECTIONS), '--data-root', str(DATA_ROOT),
                    '--output-dir', str(EVIDENCE_DIR)], check=True)
scores = json.loads(SCORES.read_text(encoding='utf-8'))
for row in sorted(scores, key=lambda row: (-row['n_exact_alignment'], row['pair']))[:12]:
    rate = row['exact_alignment_connected_rate']
    print(f"{row['pair']!r}: seen {row['n_exact_alignment']} exact times; joined rate = {rate}")

## 9. Show the small tokenizer demonstration

For the full training run, we keep only letter pairs seen at least 20 times and joined at least half the time. The tokenizer then chooses the best non-overlapping pairs in ordinary text. It uses handwriting statistics learned earlier; it does not inspect a new image at this point.

In [ ]:
from dtlr_poc.tokenizer import build_model, tokenize
from hashlib import sha256

# Use full IAM-train scores for the real tokenizer demonstration.
TRAIN_SCORES = OUTPUT_ROOT / 'iam-train-full/bigrams-dominant-core-v3/bigram_scores.json'
if TRAIN_SCORES.exists():
    raw = TRAIN_SCORES.read_bytes()
    model = build_model(json.loads(raw), sha256(raw).hexdigest(), minimum_count=20, rate_threshold=0.5)
    example = tokenize('the handwriting', model)
    print('Tokens:', example['tokens'])
    print('Pairs selected:', example['bigram_token_count'])
else:
    print('Full train scores are not available here yet. The earlier steps still provide the image walkthrough.')

## What this walkthrough demonstrates

- IAM provides the correct letters.
- DTLR gives approximate places to look.
- The image itself tells us which ink shapes touch.
- Unclear cases stay unknown instead of being forced into a wrong answer.
- Results are kept separate for training, validation, and test data.
- The final tokenizer is based on training statistics, not on a new image.